In [ ]:
import glob
import pandas as pd
from IPython.display import Markdown, display
import os

In [12]:
# parameters

annotations_path = "../../../data/dev3/annotated_usages/**/*usage_sample.tsv"
mappings_path = "../../../data/dev3/annotated_usages/**/*mappings.tsv"

s1_path = "../../../data/dev3/filtered_results/release_1/s1/result_raw.tsv.zst"
s3_path = "../../../data/dev3/filtered_results/release_1/s3/result_raw.tsv.zst"

lemur_path = "../../../data/dev3/dictionaries/lemur_dictionary.tsv"
ode_path = "../../../data/dev3/dictionaries/ode_dictionary.tsv"

result_name = s1_path.split("/")[-3]
out_dir = f"../plots/pr_table/{result_name}"  # where to save plots
os.makedirs(out_dir, exist_ok=True)

s3_thresh = None # usually the thresholds of the results is used. If you want to force a different s3 threshold, set this to the desired value e.g. 0.2. Keep in mind that setting this to a higher threshold than the one used for the results will not work correctly

In [13]:
# read annotated usages
annotation_paths = glob.glob(annotations_path, recursive=True)
df_annotation = pd.concat([pd.read_csv(a, sep="\t", dtype=str, encoding="utf-8") for a in annotation_paths])
df_annotation = df_annotation[["sense_id", "identifier", "lemma"]].rename(columns={"sense_id": "annotation_id"})

# only use first named sense_id annotation
df_annotation["annotation_id"] = df_annotation["annotation_id"].apply(lambda sense_id: sense_id.split(",")[0].strip())

# read sense_id mappings from annotated_usages
mapping_paths = glob.glob(mappings_path, recursive=True)
df_mapping = pd.concat([pd.read_csv(m, sep="\t", dtype=str, encoding="utf-8") for m in mapping_paths])

# create dictionary with sense_id mappings
mapping_dict = {}
for _,row in df_mapping.iterrows():
    mapping_dict[(row["lemma"], row["new_sense_id"])] = row["old_sense_id"]

# apply sense_id mappings to get original sense_ids from simplified ones (e.g. 1 -> LMR-1573)
df_annotation["annotation_id"] = df_annotation.apply(lambda r: mapping_dict.get((r["lemma"], r["annotation_id"]), r["annotation_id"]), axis=1)

# read ode dictionary
ode_paths = glob.glob(ode_path, recursive=True)
df_ode = pd.concat([pd.read_csv(o, sep="\t", dtype=str, encoding="utf-8") for o in ode_paths])
ode_lemmas = df_ode["lemma"].tolist()

# read lemur dictionary
lemur_paths = glob.glob(lemur_path, recursive=True)
df_lemur = pd.concat([pd.read_csv(l, sep="\t", dtype=str, encoding="utf-8") for l in lemur_paths])
df_lemur = df_lemur[["sense_id", "lemma"]].rename(columns={"sense_id": "lemur_id"})
df_lemur = df_lemur[df_lemur["lemma"].isin(set(df_annotation["lemma"].tolist()))]

# add bool column whether target word already has entries in the ODE dictionary
df_lemur["in_ode"] = df_lemur["lemma"].apply(lambda lemma: lemma in ode_lemmas)

df_annotation = df_annotation[["identifier", "lemma", "annotation_id"]]

In [14]:
df_s1 = pd.read_csv(s1_path, sep="\t", dtype={"lemma": str ,"identifier": str ,"prob": float, "thresh": float, "sense_id": str}, encoding="utf-8")[["lemma", "identifier", "prob", "thresh", "sense_id"]].rename(columns={"sense_id": "prediction_id"})
df_s3 = pd.read_csv(s3_path, sep="\t", dtype={"lemma": str ,"identifier": str ,"prob": float, "thresh": float, "sense_id": str}, encoding="utf-8")[["lemma", "identifier", "prob", "thresh", "sense_id"]].rename(columns={"sense_id": "prediction_id"})

if s3_thresh is not None:
    print(f"re-apply S3 threshold: previously: {df_s3["thresh"][0]} -> now: {s3_thresh}")
    df_s3["thresh"]=s3_thresh

df_s1 = df_s1.merge(df_annotation, on=["lemma", "identifier"], how="inner")
df_s3 = df_s3.merge(df_annotation, on=["lemma", "identifier"], how="inner")

df_s1_threshold = df_s1[df_s1["prob"]<df_s1["thresh"]].copy()
df_s3_threshold = df_s3[df_s3["prob"]<df_s3["thresh"]].copy()

# get total number of usages for each lemma at s1 (input to s1)
df_lemma_usages_s1 = df_s1["lemma"].value_counts().rename_axis("lemma").reset_index(name="lemma_usages_s1")
df_res = df_lemur.merge(df_lemma_usages_s1, how="left", on="lemma")
df_res["lemma_usages_s1"] = df_res["lemma_usages_s1"].fillna(0).astype(int)

# get total number of usages for each lemma at s3 (input to s3)
df_lemma_usages_s3 = df_s3["lemma"].value_counts().rename_axis("lemma").reset_index(name="lemma_usages_s3")
df_res = df_res.merge(df_lemma_usages_s3, how="left", on="lemma")
df_res["lemma_usages_s3"] = df_res["lemma_usages_s3"].fillna(0).astype(int)

# get total number of LEMUR sense usages for each LEMUR sense at s1 (input to s1)
df_total_lemur_senses = df_s1["annotation_id"].value_counts().rename_axis("annotation_id").reset_index(name="lemur_sense_count_s1")
df_res = df_res.merge(df_total_lemur_senses, how="left", left_on="lemur_id", right_on="annotation_id").drop(columns=["annotation_id"])
df_res["lemur_sense_count_s1"] = df_res["lemur_sense_count_s1"].fillna(0).astype(int)

# get total number of LEMUR sense usages for each LEMUR sense at s3 (input to s3)
df_total_lemur_senses = df_s3["annotation_id"].value_counts().rename_axis("annotation_id").reset_index(name="lemur_sense_count_s3")
df_res = df_res.merge(df_total_lemur_senses, how="left", left_on="lemur_id", right_on="annotation_id").drop(columns=["annotation_id"])
df_res["lemur_sense_count_s3"] = df_res["lemur_sense_count_s3"].fillna(0).astype(int)

# get number of usages predicted as LEMUR evidence for each LEMUR sense
df_predicted_lemur_count = df_s3_threshold["prediction_id"].value_counts().rename_axis("prediction_id").reset_index(name="predicted_lemur_count_s3")
df_res = df_res.merge(df_predicted_lemur_count, how="left", left_on="lemur_id", right_on="prediction_id").drop(columns=["prediction_id"])
df_res["predicted_lemur_count_s3"] = df_res["predicted_lemur_count_s3"].fillna(0).astype(int)

# get number of usages correctly labeled as LEMUR evidence for each LEMUR sense
correct_counts = df_s3_threshold[df_s3_threshold["prediction_id"] == df_s3_threshold["annotation_id"]]["prediction_id"].value_counts().rename_axis("prediction_id").reset_index(name="correct_predictions_s3")
df_res = df_res.merge(correct_counts, how="left", left_on="lemur_id", right_on="prediction_id").drop(columns=["prediction_id"])
df_res["correct_predictions_s3"] = df_res["correct_predictions_s3"].fillna(0).astype(int)

# calculate recall and precision
df_res["precision"] = df_res["correct_predictions_s3"] / df_res["predicted_lemur_count_s3"]
df_res["recall"] = df_res["correct_predictions_s3"] / df_res["lemur_sense_count_s3"]

display(Markdown("### Precision-Recall"))
print("In Macro Precision and Recall calculation ignore rows where the respective metric is NaN.")
print("E.g. 0 predictions made by the model -> no valid Precision value -> not included in Macro Precision calculation.")
print("E.g. 0 LEMUR usages in the data -> no valid Recall value -> not included in Macro Recall calculation.")

display(Markdown(f"Over all words ({len(df_res)})"))
print(f"Macro Precision over {df_res['precision'].notna().sum()} words: {df_res['precision'].mean(skipna=True):.5f}" if df_res['precision'].notna().any() else "Macro Precision: -")
print(f"Macro Recall over {df_res['recall'].notna().sum()} words: {df_res['recall'].mean(skipna=True):.5f}" if df_res['recall'].notna().any() else "Macro Recall: -")

in_ode_rows = df_res[df_res['in_ode']]
display(Markdown(f"In-ODE words only ({len(in_ode_rows)})"))
print(f"Macro Precision over {in_ode_rows['precision'].notna().sum()} words: {in_ode_rows['precision'].mean(skipna=True):.5f}" if in_ode_rows['precision'].notna().any() else "Macro Precision: -")
print(f"Macro Recall over {in_ode_rows['recall'].notna().sum()} words: {in_ode_rows['recall'].mean(skipna=True):.5f}" if in_ode_rows['recall'].notna().any() else "Macro Recall: -")

out_ode_rows = df_res[~df_res['in_ode']]
display(Markdown(f"Out-of-ODE words only ({len(out_ode_rows)})"))
print(f"Macro Precision over {out_ode_rows['precision'].notna().sum()} words: {out_ode_rows['precision'].mean(skipna=True):.5f}" if out_ode_rows['precision'].notna().any() else "Macro Precision: -")
print(f"Macro Recall over {out_ode_rows['recall'].notna().sum()} words: {out_ode_rows['recall'].mean(skipna=True):.5f}" if out_ode_rows['recall'].notna().any() else "Macro Recall: -")

df_res_copy = df_res.copy()

df_res["precision"] = df_res["precision"].round(2).fillna("-").astype(str)
df_res["recall"] = df_res["recall"].round(2).fillna("-").astype(str)

display(df_res)
df_res_copy.to_csv(f"{out_dir}/pr_table.tsv", sep="\t", encoding="utf-8", index=False)

### Precision-Recall

In Macro Precision and Recall calculation ignore rows where the respective metric is NaN.
E.g. 0 predictions made by the model -> no valid Precision value -> not included in Macro Precision calculation.
E.g. 0 LEMUR usages in the data -> no valid Recall value -> not included in Macro Recall calculation.


Over all words (48)

Macro Precision over 28 words: 0.52124
Macro Recall over 25 words: 0.51342


In-ODE words only (24)

Macro Precision over 9 words: 0.34503
Macro Recall over 7 words: 0.22953


Out-of-ODE words only (24)

Macro Precision over 19 words: 0.60471
Macro Recall over 18 words: 0.62382


,lemur_id,lemma,in_ode,lemma_usages_s1,lemma_usages_s3,lemur_sense_count_s1,lemur_sense_count_s3,predicted_lemur_count_s3,correct_predictions_s3,precision,recall
0,LMR2-42417,adoptive,True,100,50,7,7,1,1,1.0,0.14
1,LMR2-78835,prefill,True,83,65,5,5,19,2,0.11,0.4
2,LMR2-65777,kanafeh,False,30,30,30,30,25,25,1.0,0.83
3,LMR2-49106,acker,False,30,30,0,0,1,0,0.0,-
4,LMR2-53661,booby,True,100,69,47,47,3,3,1.0,0.06
5,LMR2-49027,hale,True,100,100,0,0,0,0,-,-
6,LMR2-81261,to thread the needle,False,30,30,14,14,7,3,0.43,0.21
7,LMR2-61766,air tanker,False,30,30,27,27,30,27,0.9,1.0
8,LMR2-82760,drinker,True,100,44,0,0,0,0,-,-
9,LMR2-45671,funk,True,100,19,0,0,0,0,-,-
